In [1]:
"""
PREDICCIÓN PRE-PARTIDO — BARCELONA
Combina estilo de juego (StatsBomb + Grafos) con datos
históricos de Understat para predecir resultado y xG
"""

import sys
import os
import ast
import warnings
warnings.filterwarnings('ignore')
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)
sys.path.insert(0, project_root)

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

# COmparamos los nombres de StatsBomb y Understat
NOMBRE_MAP = {
    'Atlético Madrid':   'Atletico Madrid',
    'Cádiz':             'Cadiz',
    'Deportivo Alavés':  'Alaves',
    'Levante UD':        'Levante',
    'Real Valladolid':   'Valladolid',
    'Athletic Club':     'Athletic Club',
    'Celta Vigo':        'Celta Vigo',
    'Real Betis':        'Real Betis',
    'Real Sociedad':     'Real Sociedad',
    'Real Madrid':       'Real Madrid',
    'Sevilla':           'Sevilla',
    'Valencia':          'Valencia',
    'Villarreal':        'Villarreal',
    'Getafe':            'Getafe',
    'Osasuna':           'Osasuna',
    'Granada':           'Granada',
}

print(" Librerías cargadas")
print(" PREDICCIÓN PRE-PARTIDO — BARCELONA")
print("   StatsBomb (grafos + estilo) + Understat (xG histórico)")

 Librerías cargadas
 PREDICCIÓN PRE-PARTIDO — BARCELONA
   StatsBomb (grafos + estilo) + Understat (xG histórico)


In [2]:
# CARGA DE TODOS LOS DATOS

print("\n CARGANDO DATOS")

#  Datos de Understat
path_features = os.path.join(
    project_root, 'data', 'processed', 'understat_team_features.csv'
)
path_matches = os.path.join(
    project_root, 'data', 'processed', 'understat_matches.csv'
)

df_features  = pd.read_csv(path_features)
all_features = df_features.to_dict('records')

def parse_dict_col(val):
    try:
        if isinstance(val, dict):
            return val
        return ast.literal_eval(val)
    except:
        return {}

df_all_matches = pd.read_csv(path_matches)
df_all_matches['xG_home'] = df_all_matches['xG'].apply(
    lambda x: float(parse_dict_col(x).get('h', 0))
)
df_all_matches['xG_away'] = df_all_matches['xG'].apply(
    lambda x: float(parse_dict_col(x).get('a', 0))
)
df_all_matches['goals_home'] = df_all_matches['goals'].apply(
    lambda x: int(parse_dict_col(x).get('h', 0))
)
df_all_matches['goals_away'] = df_all_matches['goals'].apply(
    lambda x: int(parse_dict_col(x).get('a', 0))
)
df_all_matches['result'] = df_all_matches.apply(
    lambda row: 'H' if row['goals_home'] > row['goals_away']
    else ('A' if row['goals_home'] < row['goals_away'] else 'D'),
    axis=1
)
df_all_matches['home_team'] = df_all_matches['h'].apply(
    lambda x: parse_dict_col(x).get('title', '')
)
df_all_matches['away_team'] = df_all_matches['a'].apply(
    lambda x: parse_dict_col(x).get('title', '')
)

print(f" Understat cargado:")
print(f"   - Features equipos: {len(all_features)} registros")
print(f"   - Partidos:         {len(df_all_matches)}")
print(f"   - Temporadas:       {sorted(df_all_matches['season'].unique().tolist())}")

#  Datos de StatsBomb (estilo Barcelona) 
path_style = os.path.join(
    project_root, 'data', 'processed', 'barcelona_style_data.csv'
)
path_style_metrics = os.path.join(
    project_root, 'data', 'processed', 'barcelona_style_metrics.csv'
)

df_barca_style   = pd.read_csv(path_style)
df_style_metrics = pd.read_csv(path_style_metrics)

print(f"\n StatsBomb (Barcelona) cargado:")
print(f"   - Partidos:  {len(df_barca_style)}")
print(f"   - Estilos:   {df_barca_style['estilo_tactico'].unique().tolist()}")
print(f"\n Win rate de Barcelona por estilo:")
for _, row in df_style_metrics.iterrows():
    print(f"   {row.name:<25}: win_rate={row['win_rate']:.3f} | xG_medio={row['xg_mean']:.3f}")


 CARGANDO DATOS
 Understat cargado:
   - Features equipos: 84 registros
   - Partidos:         2660
   - Temporadas:       [2018, 2019, 2020, 2021, 2022, 2023, 2024]

 StatsBomb (Barcelona) cargado:
   - Partidos:  102
   - Estilos:   ['posesion_alta', 'posesion_moderada', 'equilibrado']

 Win rate de Barcelona por estilo:
   0                        : win_rate=0.750 | xG_medio=1.821
   1                        : win_rate=0.690 | xG_medio=1.809
   2                        : win_rate=0.467 | xG_medio=2.130


In [3]:
# CONSTRUCCIÓN DE FEATURES COMBINADAS

print("\n CONSTRUYENDO FEATURES COMBINADAS")
print("Combinando: Understat (xG histórico) + StatsBomb (estilo/grafos)")
print("+ Head-to-head histórico + Factor de localía de Barcelona")

numeric_keys = [
    'xg_mean', 'xg_against_mean', 'xg_diff_mean',
    'goals_mean', 'goals_against_mean', 'goals_diff_mean',
    'win_rate', 'draw_rate', 'loss_rate',
    'home_win_rate', 'away_win_rate',
    'recent_xg_mean', 'recent_xg_against_mean',
    'recent_goals_mean', 'recent_win_rate',
    'recent_draw_rate', 'recent_loss_rate',
    'recent_points'
]

# Pesos por temporada
temporadas_ordenadas = sorted(df_features['season'].unique().tolist())
n_seasons      = len(temporadas_ordenadas)
season_weights = {
    season: (i + 1) / sum(range(1, n_seasons + 1))
    for i, season in enumerate(temporadas_ordenadas)
}

print(f" Pesos por temporada:")
for s, w in season_weights.items():
    print(f"   {s}/{int(s)+1}: {w:.3f}")

def get_team_aggregated_features(team_name):
    
    team_data = df_features[df_features['team'] == team_name]
    if team_data.empty:
        return {}

    aggregated = {'team': team_name}
    for key in numeric_keys:
        if key in team_data.columns:
            values, weights = [], []
            for _, row in team_data.iterrows():
                w   = season_weights.get(str(row['season']), 1.0)
                val = row.get(key, np.nan)
                if not pd.isna(val):
                    values.append(val)
                    weights.append(w)
            aggregated[key] = np.average(values, weights=weights) if values else 0.0
    return aggregated

# Features agregadas de Understat
team_agg_features = {}
for team in df_features['team'].unique():
    team_agg_features[team] = get_team_aggregated_features(team)

# Features de estilo de Barcelona (StatsBomb)
barca_style_agg = {
    'estilo_predominante':  df_barca_style['estilo_tactico'].mode()[0],
    'density_mean':         df_barca_style['static_density'].mean(),
    'clustering_mean':      df_barca_style['static_avg_clustering'].mean(),
    'betweenness_mean':     df_barca_style['static_avg_betweenness'].mean(),
    'pagerank_mean':        df_barca_style['static_avg_pagerank'].mean(),
    'pass_accuracy_mean':   df_barca_style['pass_accuracy'].mean(),
    'total_passes_mean':    df_barca_style['total_passes'].mean(),
}

barca_wr_by_style = df_style_metrics['win_rate'].to_dict()
barca_xg_by_style = df_style_metrics['xg_mean'].to_dict()

#  Head-to-head histórico 
def get_head_to_head(home_team, away_team):
    
    h2h = df_all_matches[
        (
            (df_all_matches['home_team'] == home_team) &
            (df_all_matches['away_team'] == away_team)
        ) | (
            (df_all_matches['home_team'] == away_team) &
            (df_all_matches['away_team'] == home_team)
        )
    ].copy()

    if h2h.empty:
        return {
            'h2h_matches':         0,
            'h2h_home_win_rate':   0.33,
            'h2h_draw_rate':       0.33,
            'h2h_away_win_rate':   0.33,
            'h2h_home_xg_mean':    1.5,
            'h2h_away_xg_mean':    1.0,
            'h2h_home_goals_mean': 1.5,
            'h2h_away_goals_mean': 1.0,
        }

    return {
        'h2h_matches':         len(h2h),
        'h2h_home_win_rate':   (h2h['result'] == 'H').mean(),
        'h2h_draw_rate':       (h2h['result'] == 'D').mean(),
        'h2h_away_win_rate':   (h2h['result'] == 'A').mean(),
        'h2h_home_xg_mean':    h2h['xG_home'].mean(),
        'h2h_away_xg_mean':    h2h['xG_away'].mean(),
        'h2h_home_goals_mean': h2h['goals_home'].mean(),
        'h2h_away_goals_mean': h2h['goals_away'].mean(),
    }

#  Factor de localía específico de Barcelona 
barca_home_matches = df_all_matches[
    df_all_matches['home_team'] == 'Barcelona'
]
barca_away_matches = df_all_matches[
    df_all_matches['away_team'] == 'Barcelona'
]

barca_home_stats = {
    'barca_home_win_rate':   (barca_home_matches['result'] == 'H').mean(),
    'barca_home_draw_rate':  (barca_home_matches['result'] == 'D').mean(),
    'barca_home_loss_rate':  (barca_home_matches['result'] == 'A').mean(),
    'barca_home_xg_mean':    barca_home_matches['xG_home'].mean(),
    'barca_home_xg_against': barca_home_matches['xG_away'].mean(),
}

barca_away_stats = {
    'barca_away_win_rate':   (barca_away_matches['result'] == 'A').mean(),
    'barca_away_draw_rate':  (barca_away_matches['result'] == 'D').mean(),
    'barca_away_loss_rate':  (barca_away_matches['result'] == 'H').mean(),
    'barca_away_xg_mean':    barca_away_matches['xG_away'].mean(),
    'barca_away_xg_against': barca_away_matches['xG_home'].mean(),
}

print(f"\n Barcelona como LOCAL:")
print(f"   Win rate:  {barca_home_stats['barca_home_win_rate']:.3f}")
print(f"   xG medio:  {barca_home_stats['barca_home_xg_mean']:.3f}")

print(f"\n Barcelona como VISITANTE:")
print(f"   Win rate:  {barca_away_stats['barca_away_win_rate']:.3f}")
print(f"   xG medio:  {barca_away_stats['barca_away_xg_mean']:.3f}")

print(f"\n Features de Understat: {len(team_agg_features)} equipos")
print(f" Estilo predominante Barcelona: {barca_style_agg['estilo_predominante']}")

#  Dataset de entrenamiento 
barca_matches = df_all_matches[
    (df_all_matches['home_team'] == 'Barcelona') |
    (df_all_matches['away_team'] == 'Barcelona')
].copy()

print(f"\n Partidos de Barcelona en Understat: {len(barca_matches)}")

match_dataset = []

for _, row in barca_matches.iterrows():
    home_team     = row['home_team']
    away_team     = row['away_team']
    barca_es_local = home_team == 'Barcelona'

    home_f = team_agg_features.get(home_team, {})
    away_f = team_agg_features.get(away_team, {})

    if not home_f or not away_f:
        continue

    features = {'home_advantage': 1.0 if barca_es_local else 0.0}

    # Features de Understat
    for key in numeric_keys:
        features[f'home_{key}'] = home_f.get(key, 0.0)
        features[f'away_{key}'] = away_f.get(key, 0.0)
        features[f'diff_{key}'] = (
            home_f.get(key, 0.0) - away_f.get(key, 0.0)
        )

    # Features de estilo de Barcelona (StatsBomb)
    estilo = barca_style_agg['estilo_predominante']
    features['barca_density']        = barca_style_agg['density_mean']
    features['barca_clustering']     = barca_style_agg['clustering_mean']
    features['barca_betweenness']    = barca_style_agg['betweenness_mean']
    features['barca_pagerank']       = barca_style_agg['pagerank_mean']
    features['barca_pass_accuracy']  = barca_style_agg['pass_accuracy_mean']
    features['barca_total_passes']   = barca_style_agg['total_passes_mean']
    features['barca_style_win_rate'] = barca_wr_by_style.get(estilo, 0.0)
    features['barca_style_xg_mean']  = barca_xg_by_style.get(estilo, 0.0)

    # Head-to-head
    h2h = get_head_to_head(home_team, away_team)
    features.update(h2h)

    # Factor de localía
    if barca_es_local:
        features.update(barca_home_stats)
    else:
        features.update(barca_away_stats)

    features['result']  = row['result']
    features['xg_home'] = row['xG_home']
    features['xg_away'] = row['xG_away']
    match_dataset.append(features)

df_model = pd.DataFrame(match_dataset)

print(f"\n Dataset construido: {len(df_model)} partidos de Barcelona")
print(f"\n Distribución de resultados:")
print(df_model['result'].value_counts())
print(f"\n xG medio:")
print(f"   Local:     {df_model['xg_home'].mean():.3f}")
print(f"   Visitante: {df_model['xg_away'].mean():.3f}")


 CONSTRUYENDO FEATURES COMBINADAS
Combinando: Understat (xG histórico) + StatsBomb (estilo/grafos)
+ Head-to-head histórico + Factor de localía de Barcelona
 Pesos por temporada:
   2018/2019: 0.036
   2019/2020: 0.071
   2020/2021: 0.107
   2021/2022: 0.143
   2022/2023: 0.179
   2023/2024: 0.214
   2024/2025: 0.250

 Barcelona como LOCAL:
   Win rate:  0.737
   xG medio:  2.434

 Barcelona como VISITANTE:
   Win rate:  0.602
   xG medio:  1.880

 Features de Understat: 12 equipos
 Estilo predominante Barcelona: posesion_alta

 Partidos de Barcelona en Understat: 266

 Dataset construido: 152 partidos de Barcelona

 Distribución de resultados:
result
H    69
A    55
D    28
Name: count, dtype: int64

 xG medio:
   Local:     1.776
   Visitante: 1.444


In [4]:
# ENTRENAMIENTO DE MODELOS

print("\n ENTRENANDO MODELOS...")

target_cols = ['result', 'xg_home', 'xg_away']
X = df_model.drop(columns=target_cols).select_dtypes(include=[np.number])
X = X.fillna(X.mean())

#  Resultado 
y_result = df_model['result']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_result, test_size=0.25, random_state=42, stratify=y_result
)

scaler_r      = StandardScaler()
X_train_r_sc  = scaler_r.fit_transform(X_train_r)
X_test_r_sc   = scaler_r.transform(X_test_r)

le_r          = LabelEncoder()
y_train_r_enc = le_r.fit_transform(y_train_r)
y_test_r_enc  = le_r.transform(y_test_r)

modelo_result = RandomForestClassifier(
    n_estimators=200, random_state=42, n_jobs=-1
)
modelo_result.fit(X_train_r_sc, y_train_r_enc)
acc = modelo_result.score(X_test_r_sc, y_test_r_enc)
print(f" Modelo resultado   — Accuracy: {acc:.4f}")

# xG 
y_xg_home = df_model['xg_home']
y_xg_away = df_model['xg_away']

X_train_xg, X_test_xg, y_train_xg_h, y_test_xg_h = train_test_split(
    X, y_xg_home, test_size=0.25, random_state=42
)
_, _, y_train_xg_a, y_test_xg_a = train_test_split(
    X, y_xg_away, test_size=0.25, random_state=42
)

scaler_xg     = StandardScaler()
X_train_xg_sc = scaler_xg.fit_transform(X_train_xg)
X_test_xg_sc  = scaler_xg.transform(X_test_xg)

modelo_xg_home = RandomForestRegressor(
    n_estimators=200, random_state=42, n_jobs=-1
)
modelo_xg_home.fit(X_train_xg_sc, y_train_xg_h)
modelo_xg_away = RandomForestRegressor(
    n_estimators=200, random_state=42, n_jobs=-1
)
modelo_xg_away.fit(X_train_xg_sc, y_train_xg_a)

from sklearn.metrics import mean_absolute_error
mae_h = mean_absolute_error(y_test_xg_h, modelo_xg_home.predict(X_test_xg_sc))
mae_a = mean_absolute_error(y_test_xg_a, modelo_xg_away.predict(X_test_xg_sc))

print(f" Modelo xG local    — MAE: {mae_h:.4f}")
print(f" Modelo xG visitante — MAE: {mae_a:.4f}")

# Feature importance
print(f"\n TOP 10 FEATURES MÁS IMPORTANTES (resultado):")
fi = pd.DataFrame({
    'feature':    X.columns,
    'importance': modelo_result.feature_importances_
}).sort_values('importance', ascending=False).head(10)
display(fi)

print(f" MODELOS LISTOS")


 ENTRENANDO MODELOS...
 Modelo resultado   — Accuracy: 0.5789
 Modelo xG local    — MAE: 0.8481
 Modelo xG visitante — MAE: 0.6876

 TOP 10 FEATURES MÁS IMPORTANTES (resultado):


,feature,importance
9,diff_xg_diff_mean,0.049282
18,diff_goals_diff_mean,0.039765
27,diff_loss_rate,0.039191
3,diff_xg_mean,0.036844
24,diff_draw_rate,0.030976
36,diff_recent_xg_mean,0.030292
2,away_xg_mean,0.027501
30,diff_home_win_rate,0.027449
12,diff_goals_mean,0.025679
54,diff_recent_points,0.025671


 MODELOS LISTOS


In [5]:
# FUNCIÓN DE PREDICCIÓN

result_map = {
    'H': 'Victoria Barcelona',
    'D': 'Empate',
    'A': 'Victoria Rival'
}

def predecir_partido_barca(rival_name, barca_es_local):
    """
    Predice resultado y xG de un partido de Barcelona.
    Usa Understat + StatsBomb + Head-to-head + Localía.
    """
    rival_f = team_agg_features.get(rival_name, {})
    barca_f = team_agg_features.get('Barcelona', {})

    if not rival_f or not barca_f:
        print(f" No hay datos para {rival_name}")
        return None

    home_f = barca_f  if barca_es_local else rival_f
    away_f = rival_f  if barca_es_local else barca_f

    features = {'home_advantage': 1.0 if barca_es_local else 0.0}

    for key in numeric_keys:
        features[f'home_{key}'] = home_f.get(key, 0.0)
        features[f'away_{key}'] = away_f.get(key, 0.0)
        features[f'diff_{key}'] = (
            home_f.get(key, 0.0) - away_f.get(key, 0.0)
        )

    # Features de estilo de Barcelona
    estilo = barca_style_agg['estilo_predominante']
    features['barca_density']        = barca_style_agg['density_mean']
    features['barca_clustering']     = barca_style_agg['clustering_mean']
    features['barca_betweenness']    = barca_style_agg['betweenness_mean']
    features['barca_pagerank']       = barca_style_agg['pagerank_mean']
    features['barca_pass_accuracy']  = barca_style_agg['pass_accuracy_mean']
    features['barca_total_passes']   = barca_style_agg['total_passes_mean']
    features['barca_style_win_rate'] = barca_wr_by_style.get(estilo, 0.0)
    features['barca_style_xg_mean']  = barca_xg_by_style.get(estilo, 0.0)

    # Head-to-head
    home_name = 'Barcelona' if barca_es_local else rival_name
    away_name = rival_name  if barca_es_local else 'Barcelona'
    h2h = get_head_to_head(home_name, away_name)
    features.update(h2h)

    # Factor de localía
    if barca_es_local:
        features.update(barca_home_stats)
    else:
        features.update(barca_away_stats)

    X_pred = pd.DataFrame([features]).reindex(
        columns=X.columns, fill_value=0.0
    )

    X_pred_r_sc  = scaler_r.transform(X_pred)
    X_pred_xg_sc = scaler_xg.transform(X_pred)

    result_proba  = modelo_result.predict_proba(X_pred_r_sc)[0]
    result_pred   = le_r.inverse_transform([result_proba.argmax()])[0]
    result_labels = le_r.classes_
    xg_home_pred = max(0.0, float(modelo_xg_home.predict(X_pred_xg_sc)[0]))
    xg_away_pred = max(0.0, float(modelo_xg_away.predict(X_pred_xg_sc)[0]))

    xg_barca = xg_home_pred if barca_es_local else xg_away_pred
    xg_rival = xg_away_pred if barca_es_local else xg_home_pred

    if result_pred == 'H':
        result_text = 'Victoria Barcelona' if barca_es_local else 'Victoria Rival'
    elif result_pred == 'A':
        result_text = 'Victoria Rival' if barca_es_local else 'Victoria Barcelona'
    else:
        result_text = 'Empate'

    prob_dict = {}
    for label, prob in zip(result_labels, result_proba):
        if label == 'H':
            key = 'Victoria Barcelona' if barca_es_local else 'Victoria Rival'
        elif label == 'A':
            key = 'Victoria Rival' if barca_es_local else 'Victoria Barcelona'
        else:
            key = 'Empate'
        prob_dict[key] = prob

    return {
        'barca_local':   barca_es_local,
        'rival':         rival_name,
        'xg_barca':      xg_barca,
        'xg_rival':      xg_rival,
        'result_text':   result_text,
        'probabilities': prob_dict,
        'estilo_barca':  estilo,
        'h2h_matches':   h2h['h2h_matches']
    }

def mostrar_prediccion_barca(pred, rival_display):
    if pred is None:
        return

    barca_local = pred['barca_local']
    local_name  = 'Barcelona'   if barca_local else rival_display
    visit_name  = rival_display if barca_local else 'Barcelona'

    print(f" {local_name} (local) vs {visit_name} (visitante)")
    print(f"\n xG esperado:")
    print(f"   Barcelona   : {pred['xg_barca']:.2f}")
    print(f"   {rival_display:<13}: {pred['xg_rival']:.2f}")
    print(f"\n Resultado más probable: {pred['result_text']}")
    print(f"\n Probabilidades:")
    for text, prob in pred['probabilities'].items():
        bar = '█' * int(prob * 20)
        print(f"   {text:<25}: {bar:<20} {prob*100:.1f}%")
    print(f"\n Estilo de juego Barcelona: {pred['estilo_barca']}")
    print(f" Partidos h2h históricos:   {pred['h2h_matches']}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Probabilidades
    labels_text = list(pred['probabilities'].keys())
    probs       = list(pred['probabilities'].values())
    colors      = [
        'green'     if 'Barcelona' in l
        else 'gray' if 'Empate' in l
        else 'red'
        for l in labels_text
    ]

    bars = axes[0].bar(
        labels_text, [p * 100 for p in probs],
        color=colors, edgecolor='black',
        linewidth=0.8, alpha=0.85
    )
    for bar, val in zip(bars, probs):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val*100:.1f}%', ha='center', va='bottom',
            fontsize=12, fontweight='bold'
        )
    axes[0].set_ylabel('Probabilidad (%)', fontsize=12)
    axes[0].set_title(
        f'Probabilidades de Resultado\n{local_name} vs {visit_name}',
        fontsize=13, fontweight='bold'
    )
    axes[0].set_ylim(0, 100)
    axes[0].grid(axis='y', alpha=0.3)

    # xG
    bars2 = axes[1].bar(
        ['Barcelona', rival_display],
        [pred['xg_barca'], pred['xg_rival']],
        color=['#A50044', 'steelblue'],
        edgecolor='black', linewidth=0.8, alpha=0.85
    )
    for bar, val in zip(bars2, [pred['xg_barca'], pred['xg_rival']]):
        axes[1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom',
            fontsize=14, fontweight='bold'
        )
    axes[1].set_ylabel('xG Esperado', fontsize=12)
    axes[1].set_title(
        f'Expected Goals\nBarcelona vs {rival_display}',
        fontsize=13, fontweight='bold'
    )
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle(
        f'Predicción Pre-Partido: {local_name} vs {visit_name}\n'
        f'Estilo Barcelona: {pred["estilo_barca"]} | '
        f'H2H: {pred["h2h_matches"]} partidos históricos',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()

print(" Función de predicción lista")

 Función de predicción lista


In [6]:
# SELECTOR INTERACTIVO

print("\n PREDICTOR PRE-PARTIDO — BARCELONA")

# Rivales disponibles — equipos en Understat excluyendo Barcelona

RIVALS_DISPLAY = {
    'Real Madrid':    'Real Madrid',
    'Atletico Madrid':'Atlético Madrid',
    'Sevilla':        'Sevilla',
    'Valencia':       'Valencia',
    'Villarreal':     'Villarreal',
    'Athletic Club':  'Athletic Club',
    'Real Sociedad':  'Real Sociedad',
    'Real Betis':     'Real Betis',
    'Getafe':         'Getafe',
    'Celta Vigo':     'Celta Vigo',
    'Osasuna':        'Osasuna',
}

rival_options = [(v, k) for k, v in RIVALS_DISPLAY.items()]

dropdown_rival = widgets.Dropdown(
    options=rival_options,
    value='Real Madrid',
    description=' Rival:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)

toggle_local = widgets.ToggleButtons(
    options=['Barcelona Local', 'Barcelona Visitante'],
    value='Barcelona Local',
    description='',
    button_style='info',
    layout=widgets.Layout(width='350px')
)

btn_predecir = widgets.Button(
    description=' Predecir Partido',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

output = widgets.Output()

def on_predecir(b):
    with output:
        clear_output(wait=True)

        rival_understat = dropdown_rival.value
        rival_display   = RIVALS_DISPLAY[rival_understat]
        barca_es_local  = toggle_local.value == 'Barcelona Local'

        print(f"  Analizando Barcelona vs {rival_display}")
        pred = predecir_partido_barca(rival_understat, barca_es_local)
        mostrar_prediccion_barca(pred, rival_display)

btn_predecir.on_click(on_predecir)

display(widgets.VBox([
    widgets.HTML(
        "<h3 style='color:#A50044'>🔵🔴 Predictor Pre-Partido — FC Barcelona</h3>"
        "<p style='color:gray'>Modelo entrenado con datos de Understat (7 temporadas) "
        "+ métricas de grafos de pases (StatsBomb)</p>"
    ),
    widgets.HBox([dropdown_rival, toggle_local]),
    btn_predecir,
    output
]))


 PREDICTOR PRE-PARTIDO — BARCELONA


In [7]:
# GUARDAR MODELOS
import joblib

models_dir = os.path.join(project_root, 'models')
os.makedirs(models_dir, exist_ok=True)

joblib.dump({
    'model':    modelo_result,
    'scaler':   scaler_r,
    'encoder':  le_r,
    'features': X.columns.tolist()
}, os.path.join(models_dir, 'barca_result_predictor.pkl'))

joblib.dump({
    'model_home': modelo_xg_home,
    'model_away': modelo_xg_away,
    'scaler':     scaler_xg,
    'features':   X.columns.tolist()
}, os.path.join(models_dir, 'barca_xg_predictor.pkl'))

print(" Modelos guardados:")
print(f"    models/barca_result_predictor.pkl")
print(f"    models/barca_xg_predictor.pkl")
print(f"   Predictor especializado en FC Barcelona")
print(f"   Combina StatsBomb (grafos) + Understat (xG histórico)")


 Modelos guardados:
    models/barca_result_predictor.pkl
    models/barca_xg_predictor.pkl
   Predictor especializado en FC Barcelona
   Combina StatsBomb (grafos) + Understat (xG histórico)
